# MPC Toy Demo (Model Predictive Control)

This is a compact, self-learning MPC example.

It is intentionally a toy setup (1D double integrator) so core ideas stay clear: prediction horizon, constrained control search, and receding-horizon execution.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

## 1. System model and constraints

We use a 1D double integrator with state $x=[p, v]$ and input $u$ (acceleration).

In [ ]:
dt = 0.15

# Dynamics: p_{k+1} = p_k + v_k dt ; v_{k+1} = v_k + u_k dt
def step(x, u):
    p, v = x
    p2 = p + v * dt
    v2 = v + u * dt
    return np.array([p2, v2])

# Bounds
u_min, u_max = -1.0, 1.0
v_min, v_max = -2.0, 2.0

x0 = np.array([0.0, 0.0])
x_ref = np.array([8.0, 0.0])

## 2. MPC horizon and objective

Objective balances:
- position/velocity tracking,
- control effort,
- control smoothness (input rate).

In [ ]:
N = 12              # prediction horizon
n_rollouts = 900    # sampled control sequences per MPC step

Qp = 1.8
Qv = 0.25
Ru = 0.08
Rdu = 0.12
Q_terminal = 3.0

rng = np.random.default_rng(12)

## 3. Sample-based MPC optimizer (toy approximation)

A production MPC would typically solve QP/NLP directly.
Here we sample many control sequences and pick the one with minimum constrained cost for educational clarity.

In [ ]:
def mpc_select_u(x_now, u_prev):
    best_cost = np.inf
    best_seq = None
    best_pred = None

    # Random shooting in constrained input range
    U = rng.uniform(u_min, u_max, size=(n_rollouts, N))

    for seq in U:
        x = x_now.copy()
        pred = [x.copy()]
        cost = 0.0
        feasible = True
        u_last = u_prev

        for uk in seq:
            x = step(x, uk)
            pred.append(x.copy())

            # State constraint
            if x[1] < v_min or x[1] > v_max:
                feasible = False
                break

            e = x - x_ref
            cost += Qp * e[0]**2 + Qv * e[1]**2
            cost += Ru * uk**2 + Rdu * (uk - u_last)**2
            u_last = uk

        if not feasible:
            continue

        eN = pred[-1] - x_ref
        cost += Q_terminal * (eN[0]**2 + 0.5 * eN[1]**2)

        if cost < best_cost:
            best_cost = cost
            best_seq = seq
            best_pred = np.array(pred)

    if best_seq is None:
        return 0.0, None, np.inf

    return float(best_seq[0]), best_pred, float(best_cost)

## 4. Receding-horizon closed-loop simulation

At each step we solve the horizon optimization, apply only the first input, then repeat.

In [ ]:
x = x0.copy()
u_prev = 0.0

xs = [x.copy()]
us = []
pred_bank = []
costs = []

steps = 55
for _ in range(steps):
    u_cmd, pred, J = mpc_select_u(x, u_prev)
    x = step(x, u_cmd)

    xs.append(x.copy())
    us.append(u_cmd)
    costs.append(J)
    if pred is not None:
        pred_bank.append(pred)

    u_prev = u_cmd

xs = np.array(xs)
us = np.array(us)

print('Final state:', np.round(xs[-1], 3))
print('Reference:', x_ref)

## 5. Visualize tracking and control

In [ ]:
t = np.arange(len(xs)) * dt
tu = np.arange(len(us)) * dt

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=False)

axes[0].plot(t, xs[:, 0], color='deepskyblue', label='position')
axes[0].axhline(x_ref[0], color='gray', linestyle='--', label='position ref')
axes[0].set_ylabel('position')
axes[0].legend(loc='best')

axes[1].plot(t, xs[:, 1], color='slateblue', label='velocity')
axes[1].axhline(x_ref[1], color='gray', linestyle='--', label='velocity ref')
axes[1].axhline(v_max, color='tomato', linestyle=':', alpha=0.8)
axes[1].axhline(v_min, color='tomato', linestyle=':', alpha=0.8)
axes[1].set_ylabel('velocity')
axes[1].legend(loc='best')

axes[2].step(tu, us, where='post', color='darkorange', label='u command')
axes[2].axhline(u_max, color='tomato', linestyle=':', alpha=0.8)
axes[2].axhline(u_min, color='tomato', linestyle=':', alpha=0.8)
axes[2].set_xlabel('time [s]')
axes[2].set_ylabel('accel')
axes[2].legend(loc='best')

plt.tight_layout()
plt.show()

## 6. Optional horizon preview view

Dashed curves are selected predicted trajectories at sampled control steps.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(xs[:, 0], xs[:, 1], color='deepskyblue', linewidth=2.0, label='executed state path')

if pred_bank:
    stride = max(1, len(pred_bank) // 10)
    for p in pred_bank[::stride]:
        plt.plot(p[:, 0], p[:, 1], '--', color='slateblue', alpha=0.35, linewidth=1.0)

plt.scatter([x0[0]], [x0[1]], c='limegreen', s=70, label='start')
plt.scatter([x_ref[0]], [x_ref[1]], c='crimson', s=80, marker='*', label='reference')
plt.xlabel('position')
plt.ylabel('velocity')
plt.title('Receding-horizon behavior in state space')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

## 7. Self-study experiments

Try these changes:
1. Increase horizon `N` and compare tracking smoothness vs compute time.
2. Increase `Ru` to penalize control effort and observe gentler control.
3. Tighten velocity bounds and observe constraint-active behavior.
4. Increase `n_rollouts` and check if solution quality improves.